In [ ]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# Lists to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None,  # Default to None (indicating missing content)
        "error_code": None  # New field for failed downloads
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        file_entry["error_code"] = "Invalid URL"
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        file_entry["error_code"] = response.status_code  # Store specific HTTP error code
        print(f"❌ Failed to fetch {direct_url} - HTTP {response.status_code}: {http_err}")
        failed_downloads.append(file_entry)
    except requests.exceptions.RequestException as e:
        file_entry["error_code"] = "Request Error"
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")

In [20]:
import os
import pandas as pd
import requests
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm  # Progress bar

In [4]:
json_df = pd.read_json("../data/raw/mod_files.json")

In [14]:
ant_channel = pd.read_csv("../data/pipeline/preprocessed.csv", usecols=["file_hash","new_subtype_label"]).query("new_subtype_label not in ['Z Neither','Receptor']")

In [6]:
def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`

In [17]:
json_df2 = json_df.merge(ant_channel, how="inner", on="file_hash")

In [24]:
json_df2.shape

(554, 9)

In [21]:
json_df2["new_subtype_label"].value_counts()

new_subtype_label
I K (A-type)               71
I Na (Transient)           56
I K (Delayed Rectifier)    54
I K (Ca-activated)         53
I H                        38
I Ca (L-Type HT)           38
R Glutamate (NMDA)         29
I K (M-type)               28
R GABA                     28
R Glutamate (AMPA)         26
I K (Rare)                 25
I Other (Nonspecific)      24
I Ca (T-type LT)           20
I Na (Persistent)          19
I Ca (HVA)                 12
I Other (Transporter)      12
I Na (General)             11
I Other (Leak)             10
Name: count, dtype: int64

In [27]:
import os
import pandas as pd
import requests
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm

# Base output directory
base_dir = "../data/raw/nmodl"
os.makedirs(base_dir, exist_ok=True)

# Loop through URLs in the DataFrame
for url in tqdm(json_df2["download_url"], desc="Downloading MOD files"):
    try:
        parsed_url = urlparse(url)
        file_param = parse_qs(parsed_url.query).get("file")
        
        if not file_param:
            print(f"❌ Skipping (no file= param): {url}")
            continue

        # Get relative path like "subdir/filename.mod"
        rel_path = file_param[0]
        output_path = os.path.join(base_dir, rel_path)

        # Make sure the output subdirectory exists
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # Download and save
        response = requests.get(url)
        response.raise_for_status()
        with open(output_path, "wb") as f:
            f.write(response.content)

    except Exception as e:
        print(f"❌ Error downloading {url}: {e}")
